# CTA historical service and ridership comparison

This notebook imports the eight dated CTA GTFS archives listed in the project manifest, aligns daily scheduled bus trips and trip-hours with CTA boardings by exact route and service date, then compares demand-only and service-aware regressors on the same matched rows.

Do not use the current google_transit.zip from CTA's schedule page for this historical comparison; it does not cover the ridership dates. Download each exact Mobility Database archive ID from the intern guide and put either the ZIP or its extracted folder in Google Drive under CTA_GTFS/<archive_id>/. ZIP files are checked against the guide SHA-256. For extracted folders, the notebook fingerprints the GTFS table contents but cannot verify the original ZIP bytes.

The notebook reports a retrospective development comparison through July, then evaluates a final fixed-origin August holdout. The August forecast is issued at the end of July, freezes the July 11 schedule archive, and feeds predictions, not actual August rides, into future demand lags. August labels are used only for scoring.

In [1]:
from pathlib import Path
import subprocess
from google.colab import drive, files
drive.mount('/content/drive')
REPO = Path('/content/Powered-Public-Transit-Route-Optimization')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'users/bidhubhusan',
                    'https://github.com/springboardmentor441p-coderr/Powered-Public-Transit-Route-Optimization.git', str(REPO)], check=True)
else:
    # Colab can keep /content and imported modules between notebook opens.
    # Refresh this source file before importing the evaluator.
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'users/bidhubhusan'], check=True)
    latest_source = subprocess.check_output([
        'git', '-C', str(REPO), 'show', 'FETCH_HEAD:src/compare_service_models.py'
    ])
    (REPO / 'src/compare_service_models.py').write_bytes(latest_source)
%cd /content/Powered-Public-Transit-Route-Optimization
%pip -q install -r requirements.txt

Mounted at /content/drive
/content/Powered-Public-Transit-Route-Optimization


## 1. Locate ridership and archived schedules

The ridership CSV downloads from the official portal if it is not already in the Colab session. Keep the eight schedule snapshots in separate folders under the Drive folder shown below.

In [2]:
import json
import pandas as pd
from urllib.request import urlretrieve
import importlib
import src.gtfs_service as gtfs_service
import src.compare_service_models as comparison_module
importlib.reload(gtfs_service)
importlib.reload(comparison_module)
build_service_table = gtfs_service.build_service_table
compare = comparison_module.compare
walk_forward_hybrid_development = comparison_module.walk_forward_hybrid_development

RIDERSHIP_PATH = Path('/content/cta_bus_ridership.csv')
if not RIDERSHIP_PATH.exists():
    urlretrieve('https://data.cityofchicago.org/api/views/jyb9-n7fm/rows.csv?accessType=DOWNLOAD', RIDERSHIP_PATH)
GTFS_DRIVE_DIR = Path('/content/drive/MyDrive/CTA_GTFS')
manifest = pd.read_csv('data/raw/gtfs_archive_manifest.csv', dtype='string').fillna('')
def archive_location(archive_id):
    folder = GTFS_DRIVE_DIR / archive_id
    zipped = folder / 'original.zip'
    return str(zipped if zipped.is_file() else folder)
manifest['zip_path'] = manifest['archive_id'].map(archive_location)
MANIFEST_PATH = Path('/content/gtfs_archive_manifest_colab.csv')
manifest.to_csv(MANIFEST_PATH, index=False)
missing = [row.zip_path for row in manifest.itertuples() if not Path(row.zip_path).exists()]
if missing:
    raise FileNotFoundError('Place all eight ZIPs or extracted archive folders in Drive first. Missing:\n' + '\n'.join(missing))
print('Ridership file:', RIDERSHIP_PATH)
print('GTFS archive count:', len(manifest))

Ridership file: /content/cta_bus_ridership.csv
GTFS archive count: 8


## 2. Import dated service and audit the exact join

The importer applies calendar_dates additions/removals, chooses the latest archive retrieved on or before each date for the whole network, and leaves unmatched rows visible. It also writes an August schedule using the latest archive available at the July 31 forecast origin. Stop-time tables are streamed, so this step can take several minutes.

In [3]:
SERVICE_PATH = Path('data/processed/cta_route_date_demand_service.csv')
FIXED_SERVICE_PATH = Path('data/processed/cta_service_fixed_origin.csv')
IMPORT_REPORT = Path('outputs/reports/gtfs_service_import.json')
import_report = build_service_table(
    RIDERSHIP_PATH, MANIFEST_PATH, SERVICE_PATH, IMPORT_REPORT,
    fixed_origin_date='2026-07-31', fixed_start_date='2026-08-01',
    fixed_end_date='2026-08-31', fixed_output_path=FIXED_SERVICE_PATH)
print(pd.Series({k: v for k, v in import_report.items() if not isinstance(v, (dict, list))}))
print('Archive and date coverage details:')
display(pd.DataFrame(import_report['archive_versions']))
print('Unmatched ridership rows:', import_report['ridership_without_schedule_rows'])
print('Unmatched schedule rows:', import_report['schedule_without_ridership_rows'])

ridership_sha256                   fa687b55d4049c9330e8088c6b79947cabfc46e010d89e...
schedule_route_date_rows                                                       31923
matched_route_date_rows                                                        31896
ridership_without_schedule_rows                                              1092274
schedule_without_ridership_rows                                                   27
service_hours_policy               Null unless every counted trip for route/date ...
archive_selection_policy           Globally select latest retrieval date on or be...
snapshot_selection_policy                                  on_or_before_service_date
output_csv                          data/processed/cta_route_date_demand_service.csv
dtype: object
Archive and date coverage details:


,archive_id,retrieved_date,archive_sha256,archive_sha256_verified,source_kind,source_fingerprint,bus_routes,bus_trip_templates
0,mdb-389-202511230128,2025-11-23,None,False,extracted_directory,c888012db87939626005b6cb4bf8b48ceee39d7e7f00be...,123,89536
1,mdb-389-202512190112,2025-12-19,None,False,extracted_directory,86f99c8d04254550e1ef98dbbfcc90705c5997992af69d...,123,89646
2,mdb-389-202601080131,2026-01-08,None,False,extracted_directory,07c0cc22eccc9eaeb8dc8328e03a9411223254de1e7fef...,123,44878
3,mdb-389-202602100132,2026-02-10,None,False,extracted_directory,e707dfbb2c2c32ceaba51aee659e7e1b74dd873a8a1750...,123,44878
4,mdb-389-202603260020,2026-03-26,None,False,extracted_directory,394f2a11d1ce948ed1925b26eac610741993af9412b667...,123,89438
5,mdb-389-202604240144,2026-04-24,None,False,extracted_directory,5798f83b844a193b26f16297239acbb9d0530592cf20f7...,123,44560
6,mdb-389-202605300003,2026-05-30,None,False,extracted_directory,3fc702195ecc6fdcb28794f922f56925219e0b3cc30a75...,125,89367
7,mdb-389-202607110020,2026-07-11,None,False,extracted_directory,6415e0c25ea1206c1b2155fb2045ba8e6d8f8578123129...,125,89614


Unmatched ridership rows: 1092274
Unmatched schedule rows: 27


## 3. Compare demand-only and service-aware models

June-July are development dates. The fixed-origin August comparison fits through July 31, freezes the July 11 archive for every August date, and recursively uses model predictions after the July 31 origin. The evaluation code never uses actual August rides as lag features. Model settings stay unchanged from the earlier comparison; compare both fitted models with the same recursive past-four-same-weekdays baseline. Service-hour values remain missing when trip durations are incomplete; the pipeline imputes them from its training data and adds missingness indicators.

In [4]:
COMPARISON_REPORT = Path('outputs/reports/service_model_comparison.json')
comparison = compare(
    RIDERSHIP_PATH, SERVICE_PATH, COMPARISON_REPORT, Path('models'),
    train_start='2025-11-23', train_end='2026-03-31',
    validation_start='2026-04-01', validation_end='2026-07-31',
    test_start='2026-08-01', test_end='2026-08-31',
    fixed_origin_service_path=FIXED_SERVICE_PATH, fixed_origin='2026-07-31')
print('August matched test coverage:', comparison['matched_test_coverage_pct'], '%')
print('Seasonal baselines on matched test rows:')
display(pd.DataFrame(comparison['seasonal_baselines_on_matched_test_rows']).T)
print('Model comparison:')
display(pd.DataFrame({name: result['test_metrics'] for name, result in comparison['models'].items()}).T)
fixed = comparison['fixed_origin_evaluation']
print('Fixed-origin August results (primary holdout):')
display(pd.DataFrame({name: result['metrics'] for name, result in fixed['models'].items()}).T)
print('Day-type breakdown: model and seasonal baseline MAE')
daytype_rows = []
for label in sorted(set(fixed['models']['service_aware']['metrics_by_daytype']) | set(fixed['seasonal_baselines_by_daytype']['past_four_same_weekdays'])):
    row = {'daytype': label}
    for model_name, result in fixed['models'].items():
        row[f'{model_name}_mae'] = result['metrics_by_daytype'].get(label, {}).get('mae')
    row['past_four_same_weekdays_mae'] = fixed['seasonal_baselines_by_daytype']['past_four_same_weekdays'].get(label, {}).get('mae')
    daytype_rows.append(row)
display(pd.DataFrame(daytype_rows).set_index('daytype'))
print('Routes where service-aware differs most from the past-four-weekday baseline (positive delta = worse):')
route_delta = pd.DataFrame(fixed['model_mae_comparison_by_route']['service_aware_vs_past_four_same_weekdays']).T
display(route_delta.sort_values('mae_delta', ascending=False).head(10))
print('Routes with the largest service-aware gains over the baseline:')
display(route_delta.sort_values('mae_delta').head(10))
print('Full route/day-type diagnostics, coverage, and source fingerprints are in the JSON report.')

print('Walk-forward hybrid development (monthly refit; policy learned only from prior folds):')
hybrid_development = walk_forward_hybrid_development(RIDERSHIP_PATH, SERVICE_PATH)
comparison['hybrid_walk_forward_development'] = hybrid_development
COMPARISON_REPORT.write_text(json.dumps(comparison, indent=2) + '\n')
display(pd.DataFrame(hybrid_development['overall_metrics']).T)
print('Hybrid and candidate MAE by day type:')
display(pd.DataFrame({name: {daytype: scores[name]['mae'] for daytype, scores in hybrid_development['metrics_by_daytype'].items()} for name in hybrid_development['overall_metrics']}).T)
print('Hybrid fold scores (these are rolling one-step development results, not long-range recursive scores):')
display(pd.DataFrame({fold['forecast_window'][0]: fold['metrics']['hybrid'] for fold in hybrid_development['folds']}).T)
print('August has already been inspected; treat all these folds as development evidence, not a fresh final holdout.')

August matched test coverage: 98.76612450925407 %
Seasonal baselines on matched test rows:


,mae,rmse,wape
last_week_same_day,525.243896,846.385850,0.106674
past_four_same_weekdays,424.769591,708.317755,0.086269


Model comparison:


,mae,rmse,wape
demand_only,398.203675,647.154226,0.080873
service_aware,392.632604,636.138528,0.079742


Fixed-origin August results (primary holdout):


,mae,rmse,wape
demand_only,489.811764,813.245089,0.099478
service_aware,468.874822,771.590207,0.095226


... (9 outputs left out)

,A,U,W
past_four_same_weekdays,385.894576,453.970850,381.502288
last_week_same_day,491.920100,540.522709,435.786313
demand_only,378.908016,415.609279,382.046207
service_aware,380.615019,401.181994,371.838665
hybrid,392.041498,434.599522,379.106753


Hybrid fold scores (these are rolling one-step development results, not long-range recursive scores):


,mae,rmse,wape
2026-03-01,410.198131,668.167183,0.084838
2026-04-01,382.490929,587.170730,0.073572
2026-05-01,324.141785,604.830106,0.061474
2026-06-01,447.529625,732.034354,0.089820
2026-07-01,353.398529,617.369041,0.072486
2026-08-01,405.664156,657.838558,0.082388


August has already been inspected; treat all these folds as development evidence, not a fresh final holdout.


## 4. Download the results

Save these reports with the project. The two fitted artifacts were trained on the same matched partitions and differ only in whether scheduled trips/service-hours are included.

In [5]:
files.download(str(IMPORT_REPORT))
files.download(str(COMPARISON_REPORT))
files.download('models/cta_demand_only.joblib')
files.download('models/cta_service_aware.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>